In [1]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import polars.selectors as cs
import pandas as pd
import plotly.express as px

import ipaddress
import datetime as dt

import polars.selectors as cs

In [2]:
df_path = r'/mnt/samsung/Datasets/spotify_2026_scenarios.csv'

In [3]:
df = pl.read_csv(df_path)

In [4]:
df.collect_schema()

Schema([('artist_name', String),
        ('track_id', String),
        ('track_name', String),
        ('acousticness', Float64),
        ('danceability', Float64),
        ('duration_ms', Float64),
        ('energy', Float64),
        ('instrumentalness', Float64),
        ('key', Int64),
        ('liveness', Float64),
        ('loudness', Float64),
        ('mode', Int64),
        ('speechiness', Float64),
        ('tempo', Float64),
        ('time_signature', Int64),
        ('valence', Float64),
        ('popularity', Int64),
        ('scenario', String),
        ('ai_generated', Int64),
        ('short_form', Int64)])

In [9]:
df.estimated_size('mb')

72.11447143554688

In [18]:
df = df.with_columns(
    pl.col('duration_ms').cast(pl.Duration('ms')).alias('duration_ms'),
)

In [23]:
numeric_cols = df.select(cs.numeric()).columns

df = df.with_columns([
    df[col].shrink_dtype() for col in numeric_cols
])

In [25]:
df.collect_schema()

Schema([('artist_name', String),
        ('track_id', String),
        ('track_name', String),
        ('acousticness', Float32),
        ('danceability', Float32),
        ('duration_ms', Duration(time_unit='ms')),
        ('energy', Float32),
        ('instrumentalness', Float32),
        ('key', Int8),
        ('liveness', Float32),
        ('loudness', Float32),
        ('mode', Int8),
        ('speechiness', Float32),
        ('tempo', Float32),
        ('time_signature', Int8),
        ('valence', Float32),
        ('popularity', Int8),
        ('scenario', String),
        ('ai_generated', Int8),
        ('short_form', Int8)])

In [26]:
df.null_count()

artist_name,track_id,track_name,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,popularity,scenario,ai_generated,short_form
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [45]:
# df.filter(
#     pl.col('track_name').is_null()
# )

null = df.null_count().unpivot(variable_name='column', value_name='nulls')
null.filter(
    pl.col('nulls') > 0
)

column,nulls
str,u32
"""track_name""",3


In [40]:
df.select(
    'popularity', 'tempo', 'loudness', 'duration_ms'
).describe().filter(
    pl.col('statistic').str.contains('min') |
    pl.col('statistic').str.contains('max')
)

statistic,popularity,tempo,loudness,duration_ms
str,f64,f64,f64,str
"""min""",0.0,0.0,-60.0,"""0:00:30"""
"""max""",100.0,249.983002,1.806,"""1:33:20.019000"""


In [32]:
df.describe()

statistic,artist_name,track_id,track_name,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,popularity,scenario,ai_generated,short_form
str,str,str,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64
"""count""","""391989""","""391989""","""391986""",391989.0,391989.0,"""391989""",391989.0,391989.0,391989.0,391989.0,391989.0,391989.0,391989.0,391989.0,391989.0,391989.0,391989.0,"""391989""",391989.0,391989.0
"""null_count""","""0""","""0""","""3""",0.0,0.0,"""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""",0.0,0.0
"""mean""",null,null,null,0.386832,0.611376,"""0:03:17.813000""",0.543341,0.329541,5.231894,0.155362,-9.974006,0.607739,0.112015,119.473351,3.878986,0.43963,24.208988,null,0.333333,0.447158
"""std""",null,null,null,0.340871,0.195139,null,0.262001,0.350515,3.602692,0.169261,6.544363,0.488255,0.124327,30.159559,0.514402,0.259079,19.713141,null,0.471405,0.497201
"""min""","""!!!""","""0009UBVA8DCDwk1Hepib6P""","""!!!!""",0.0,0.0,"""0:00:30""",0.0,0.0,0.0,0.0,-60.0,0.0,0.0,0.0,0.0,0.0,0.0,"""ai_2026""",0.0,0.0
"""25%""",null,null,null,0.102657,0.485187,"""0:02:28.728000""",0.3654,0.121082,2.0,0.053355,-11.898,0.0,0.0389,96.014,4.0,0.224,7.0,null,0.0,0.0
"""50%""",null,null,null,0.257005,0.633478,"""0:03:06.900000""",0.576032,0.185773,5.0,0.092528,-7.979,1.0,0.0559,120.027,4.0,0.42,22.0,null,0.0,0.0
"""75%""",null,null,null,0.683763,0.75848,"""0:03:46.334000""",0.749372,0.548441,8.0,0.198015,-5.684,1.0,0.129,139.641998,4.0,0.638,38.0,null,1.0,1.0
"""max""","""ｊａｒｊａｒｊｒ""","""7zzi9vwI9Oxv738nSig19o""","""흰 Whiteness""",1.0,1.0,"""1:33:20.019000""",1.0,1.0,11.0,1.0,1.806,1.0,0.966,249.983002,5.0,1.0,100.0,"""mainstream_2026""",1.0,1.0


In [41]:
df.shape

(391989, 20)

In [50]:
df.select(
    pl.col('artist_name').n_unique(),
    pl.col('scenario').n_unique(),
).unpivot()

variable,value
str,u32
"""artist_name""",34621
"""scenario""",3


In [52]:
df['scenario'].value_counts()

scenario,count
str,u32
"""chill_2026""",130663
"""ai_2026""",130663
"""mainstream_2026""",130663


In [53]:
df['ai_generated'].value_counts()

ai_generated,count
i8,u32
1,130663
0,261326


In [54]:
df['short_form'].value_counts()

short_form,count
i8,u32
0,216708
1,175281


In [55]:
for col in ['scenario', 'ai_generated', 'short_form']:
    print(f'\n--- {col} ---\n')
    print(df[col].value_counts())


--- scenario ---

shape: (3, 2)
┌─────────────────┬────────┐
│ scenario        ┆ count  │
│ ---             ┆ ---    │
│ str             ┆ u32    │
╞═════════════════╪════════╡
│ mainstream_2026 ┆ 130663 │
│ chill_2026      ┆ 130663 │
│ ai_2026         ┆ 130663 │
└─────────────────┴────────┘

--- ai_generated ---

shape: (2, 2)
┌──────────────┬────────┐
│ ai_generated ┆ count  │
│ ---          ┆ ---    │
│ i8           ┆ u32    │
╞══════════════╪════════╡
│ 1            ┆ 130663 │
│ 0            ┆ 261326 │
└──────────────┴────────┘

--- short_form ---

shape: (2, 2)
┌────────────┬────────┐
│ short_form ┆ count  │
│ ---        ┆ ---    │
│ i8         ┆ u32    │
╞════════════╪════════╡
│ 1          ┆ 175281 │
│ 0          ┆ 216708 │
└────────────┴────────┘


In [57]:
df.group_by(
    'ai_generated'
).agg(
    pl.len().alias('count')
).with_columns(
    ((pl.col('count') / pl.col('count').sum()) * 100).round(2).alias('percent')
).sort(
    'ai_generated'
)

ai_generated,count,percent
i8,u32,f64
0,261326,66.67
1,130663,33.33


In [61]:
df = df.with_columns(
    pl.col('duration_ms').dt.total_seconds().alias('duration_seconds'),
    pl.col('duration_ms').dt.total_minutes().alias('duration_minutes'),
)

In [66]:
df.select(
    pl.col('duration_minutes').mean().alias('duration_minutes_mean'),
    pl.col('duration_minutes').median().alias('duration_minutes_median'),
    pl.col('duration_minutes').max().alias('duration_minutes_max'),
).unpivot()

variable,value
str,f64
"""duration_minutes_mean""",2.795257
"""duration_minutes_median""",3.0
"""duration_minutes_max""",93.0


In [71]:
df.filter(
    pl.col('duration_ms') < pl.duration(seconds=30)
).height

0

In [76]:
df.select(
    pl.col('key').count().alias('key_count'),
    pl.col('key').n_unique().alias('key_unique'),
)

key_count,key_unique
u32,u32
391989,12


In [77]:
df['key'].unique()

key
i8
0
1
2
3
4
…
7
8
9


In [80]:
encoded_key = {
    0: 'C',
    1: 'C#',
    2: 'D',
    3: 'D#',
    4: 'E',
    5: 'F',
    6: 'F#',
    7: 'G',
    8: 'G#',
    9: 'A',
    10: 'A#',
    11: 'B',
    -1: 'No Key Detected'
}

In [92]:
key_encoded_column = df.select(
    pl.col('key').replace_strict(encoded_key, return_dtype=pl.String).alias('key_encoded')
).to_series()

df.insert_column(
    index=9, column=key_encoded_column,
)

artist_name,track_id,track_name,acousticness,danceability,duration_ms,energy,instrumentalness,key,key_encoded,liveness,loudness,mode,speechiness,tempo,time_signature,valence,popularity,scenario,ai_generated,short_form,duration_seconds,duration_minutes
str,str,str,f32,f32,duration[ms],f32,f32,i8,str,f32,f32,i8,f32,f32,i8,f32,i8,str,i8,i8,i64,i64
"""YG""","""2RM4jf1Xa9zPgMGRDiht8O""","""Big Bank feat. 2 Chainz, Big S…",0.0,0.832934,3m 38s 373ms,0.371027,0.0,1,"""C#""",0.029206,-7.678,1,0.409,203.927002,4,0.118,15,"""mainstream_2026""",0,0,218,3
"""YG""","""1tHDG53xJNGsItRA3vfVgs""","""BAND DRUM (feat. A$AP Rocky)""",0.0,0.923235,3m 14s 800ms,0.576483,0.0,8,"""G#""",0.308062,-7.259,1,0.457,159.009003,4,0.371,0,"""mainstream_2026""",0,0,194,3
"""R3HAB""","""6Wosx2euFPMT14UXiWudMy""","""Radio Silence""",0.0,0.695954,1m 58s 913ms,0.764094,0.0,9,"""A""",0.044915,-5.89,0,0.0454,114.966003,4,0.382,56,"""mainstream_2026""",0,1,118,1
"""Chris Cooq""","""3J2Jpw61sO7l6Hc7qdYV91""","""Lactose""",0.0,0.910461,1m 45s 381ms,0.606327,0.891134,5,"""F""",0.06142,-12.118,0,0.0701,123.002998,4,0.641,0,"""mainstream_2026""",0,1,105,1
"""Chris Cooq""","""2jbYvQCyPgX3CdmAzeVeuS""","""Same - Original mix""",0.0,0.858317,1m 44s 16ms,0.8251,0.860099,7,"""G""",0.0,-10.277,1,0.0661,120.046997,4,0.928,0,"""mainstream_2026""",0,1,104,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Calum Scott""","""0cvfSKcm9VeduwyYPrxtLx""","""Come Back Home""",0.072269,0.641195,2m 55s 538ms,0.78242,0.212537,11,"""B""",0.05181,-5.174,1,0.0323,131.048996,4,0.289,57,"""ai_2026""",1,1,175,2
"""Saint Claire""","""43MP9F7UzvfilSrw2SqZGJ""","""Enough for You""",0.974612,0.377441,2m 59s 583ms,0.23257,0.204567,9,"""A""",0.039139,-13.233,1,0.0437,94.039001,4,0.346,60,"""ai_2026""",1,1,179,2
"""Mike Stud""","""4TWlUuFk81NGUNKwndyS5Q""","""Do It""",0.349649,0.723665,2m 4s 190ms,0.479938,0.208852,8,"""G#""",0.055226,-8.351,0,0.206,156.977005,4,0.546,47,"""ai_2026""",1,1,124,2


In [93]:
{col for col in enumerate(df.columns)}

{(0, 'artist_name'),
 (1, 'track_id'),
 (2, 'track_name'),
 (3, 'acousticness'),
 (4, 'danceability'),
 (5, 'duration_ms'),
 (6, 'energy'),
 (7, 'instrumentalness'),
 (8, 'key'),
 (9, 'key_encoded'),
 (10, 'liveness'),
 (11, 'loudness'),
 (12, 'mode'),
 (13, 'speechiness'),
 (14, 'tempo'),
 (15, 'time_signature'),
 (16, 'valence'),
 (17, 'popularity'),
 (18, 'scenario'),
 (19, 'ai_generated'),
 (20, 'short_form'),
 (21, 'duration_seconds'),
 (22, 'duration_minutes')}